# Phase 2 Patch — Fix `level` Metadata in Chunks

**Problem:** The `level` field (1/2/3) was not propagated from the document
manifest into chunks during Phase 2. Every chunk has `level=0` as a result,
which breaks level-based metadata filtering in Phase 3.

**This patch does two things:**
1. Reads the existing `dora_chunks.json` produced by Phase 2
2. Looks up the correct `level` for each chunk from the manifest
3. Writes a corrected `dora_chunks.json` in place

**After running this patch:**
- Open `03_indexing_v2.ipynb`
- Set `FORCE_REBUILD = True` in Cell 2
- Run all cells to rebuild the index with correct level metadata
- Then set `FORCE_REBUILD = False` again for future runs

In [4]:
# Cell 1 — Load manifest and chunks
import json
from pathlib import Path

BASE_DIR    = Path("./dora_rag")
CLEANED_DIR = BASE_DIR / "dora_cleaned"
CHUNKS_DIR  = BASE_DIR / "dora_chunks"
CHUNKS_FILE = CHUNKS_DIR / "dora_chunks.json"

# Load manifest (produced by Phase 1)
manifest_path = CLEANED_DIR / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}\nRun Phase 1 first.")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

# Build lookup: source_doc name → level
level_lookup = {entry["name"]: entry["level"] for entry in manifest}

print(f"Manifest loaded: {len(manifest)} documents")
print(f"\nLevel lookup:")
for name, level in sorted(level_lookup.items()):
    print(f"  L{level}  {name}")

# Load existing chunks
if not CHUNKS_FILE.exists():
    raise FileNotFoundError(f"Chunks file not found: {CHUNKS_FILE}\nRun Phase 2 first.")

chunks = json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))
print(f"\nChunks loaded: {len(chunks)} total")

Manifest loaded: 16 documents

Level lookup:
  L1  L1_DORA_Directive_2022_2556
  L1  L1_DORA_Regulation_2022_2554
  L2  L2_DR_CTPP_Designation_Criteria
  L2  L2_DR_DORA_Oversight_Fees
  L2  L2_ITS_ICT_Incident_Reporting
  L2  L2_ITS_Register_of_Information
  L2  L2_RTS_ICT_Incident_Classification
  L2  L2_RTS_ICT_Incident_Reporting_Process
  L2  L2_RTS_ICT_Risk_Management
  L2  L2_RTS_ICT_Third_Party_Policy
  L2  L2_RTS_Joint_Examination_Team
  L2  L2_RTS_Oversight_Conditions
  L2  L2_RTS_Subcontracting
  L2  L2_RTS_TLPT
  L3  L3_ESAs_Public_Statement_DORA
  L3  L3_Guidelines_Costs_and_Losses

Chunks loaded: 390 total


In [5]:
# Cell 2 — Diagnose current level values
from collections import Counter

current_levels = Counter(c.get("level", "MISSING") for c in chunks)
print("Current level distribution in chunks:")
for lv, count in sorted(current_levels.items(), key=lambda x: str(x[0])):
    status = "✗ WRONG" if lv in (0, "MISSING") else "✓ OK"
    print(f"  level={lv}: {count} chunks  {status}")

# Check which source_docs appear in chunks but not in manifest
chunk_sources  = set(c.get("source_doc", "") for c in chunks)
missing_in_map = chunk_sources - set(level_lookup.keys())
if missing_in_map:
    print(f"\n⚠ Source docs in chunks not found in manifest: {missing_in_map}")
    print("  These will remain at level=0 after patch.")
else:
    print(f"\n✓ All {len(chunk_sources)} source docs found in manifest — patch will fix all chunks.")

Current level distribution in chunks:
  level=MISSING: 390 chunks  ✗ WRONG

✓ All 16 source docs found in manifest — patch will fix all chunks.


In [6]:
# Cell 3 — Apply level fix to all chunks
fixed   = 0
skipped = 0
unknown = 0

for chunk in chunks:
    src = chunk.get("source_doc", "")
    if src in level_lookup:
        correct_level = level_lookup[src]
        if chunk.get("level") != correct_level:
            chunk["level"] = correct_level
            fixed += 1
        else:
            skipped += 1  # already correct
    else:
        chunk["level"] = 0  # unknown — leave as 0
        unknown += 1

print(f"Chunks fixed   : {fixed}")
print(f"Already correct: {skipped}")
print(f"Unknown source : {unknown}")

# Verify result
new_levels = Counter(c.get("level", "MISSING") for c in chunks)
print(f"\nLevel distribution after fix:")
for lv, count in sorted(new_levels.items(), key=lambda x: str(x[0])):
    print(f"  level={lv}: {count} chunks")

Chunks fixed   : 390
Already correct: 0
Unknown source : 0

Level distribution after fix:
  level=1: 132 chunks
  level=2: 248 chunks
  level=3: 10 chunks


In [7]:
# Cell 4 — Save corrected chunks file
import shutil

# Back up original before overwriting
backup_path = CHUNKS_DIR / "dora_chunks_backup.json"
shutil.copy(CHUNKS_FILE, backup_path)
print(f"✓ Backup saved: {backup_path}")

# Write corrected file
CHUNKS_FILE.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✓ Corrected chunks saved: {CHUNKS_FILE}")
print(f"  Total chunks: {len(chunks)}")

# Spot-check a few chunks from each level
print(f"\nSpot-check (first chunk per level):")
seen = set()
for c in chunks:
    lv = c.get("level")
    if lv not in seen:
        seen.add(lv)
        print(f"  L{lv} | {c['source_doc'][:40]:<40} | chunk_id: {c['chunk_id'][:40]}")
    if len(seen) >= 4:
        break

✓ Backup saved: dora_rag/dora_chunks/dora_chunks_backup.json
✓ Corrected chunks saved: dora_rag/dora_chunks/dora_chunks.json
  Total chunks: 390

Spot-check (first chunk per level):
  L1 | L1_DORA_Regulation_2022_2554             | chunk_id: L1_DORA_Regulation_2__art101__p00
  L2 | L2_RTS_ICT_Risk_Management               | chunk_id: L2_RTS_ICT_Risk_Mana__art001__p00
  L3 | L3_Guidelines_Costs_and_Losses           | chunk_id: L3_Guidelines_Costs___preamble__p00


In [8]:
# Cell 5 — Instructions for next step
print("=" * 60)
print("PATCH COMPLETE")
print("=" * 60)
print("""
The level field is now correctly set in dora_chunks.json.

Next steps:
  1. Open 03_indexing_v2.ipynb
  2. In Cell 2, set:  FORCE_REBUILD = True
  3. Run all cells — the index will be rebuilt with correct metadata
  4. After indexing completes, set FORCE_REBUILD = False again

The level-based filter tests should then return results:
  Level 1 filter → regulation + directive chunks
  Level 2 filter → RTS + ITS + delegated_regulation chunks
  Level 3 filter → guideline + statement chunks
""")

PATCH COMPLETE

The level field is now correctly set in dora_chunks.json.

Next steps:
  1. Open 03_indexing_v2.ipynb
  2. In Cell 2, set:  FORCE_REBUILD = True
  3. Run all cells — the index will be rebuilt with correct metadata
  4. After indexing completes, set FORCE_REBUILD = False again

The level-based filter tests should then return results:
  Level 1 filter → regulation + directive chunks
  Level 2 filter → RTS + ITS + delegated_regulation chunks
  Level 3 filter → guideline + statement chunks

